# SARM Reward Model 預測視覺化（Lite 版）

精簡版：載入訓練好的 SARM 模型，對指定 episodes 跑 in-process inference，
輸出「左影片 + 右進度條/曲線」的 MP4。

**主要差異 vs. 完整版**
- 移除 PNG 視覺化、模型 repo 預檢查、診斷 cell（Cell 13z）
- 兩次資源載入合併為一次（節省約一半模型載入時間）

**修改 EVAL_EPISODES 即可換要 eval 的 episode（見 Cell 1）。**


In [ ]:
# Cell 0: GPU 確認
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), '錯誤：沒有 GPU！請檢查 Runtime 設定'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# Cell 1: 使用者設定
# ════════════════════════════════════════════════════════════
# 想換 eval episode 只改下面這一行即可
EVAL_EPISODES = [15, 17, 52, 57, 270, 267]
# ════════════════════════════════════════════════════════════

HF_USERNAME = 'Lebruhbruh'
HF_TOKEN    = ''   # <-- 貼上你的 HF Token（read 權限即可）

ANNOTATED_DATASET = 'Lebruhbruh/SARM-opendata-annotated-fixed'
MODEL_REPO_ID     = f'{HF_USERNAME}/sarm-charging-bimanual'

VIDEO_KEY  = 'observation.images.top'
HEAD_MODE  = 'dense'                 # 'sparse' or 'dense'
VIDEO_STRIDE = 5                     # 每 N frame 推論 1 次；改 1 = 每 frame
VIDEO_FPS    = 30
VIZ_VIDEO_DIR = '/content/sarm_predictions_video'

print(f'標注資料集: {ANNOTATED_DATASET}')
print(f'SARM 模型:  {MODEL_REPO_ID}')
print(f'相機:       {VIDEO_KEY}')
print(f'EVAL_EPISODES: {EVAL_EPISODES}')
print(f'head-mode: {HEAD_MODE}, stride: {VIDEO_STRIDE}, fps: {VIDEO_FPS}')
print(f'輸出目錄: {VIZ_VIDEO_DIR}')


In [ ]:
# Cell 1b: 進度回饋工具（後續長時 cell 共用）
import threading, time, contextlib
from datetime import timedelta

def _fmt_elapsed(seconds: float) -> str:
    return str(timedelta(seconds=int(seconds)))

@contextlib.contextmanager
def progress_stage(title: str, steps=None, heartbeat_seconds: int = 30):
    print(f'\n━━━ {title} ━━━')
    if steps:
        for i, step in enumerate(steps, 1):
            print(f'  {i}) {step}')
        print()
    stop_evt = threading.Event()
    start = time.monotonic()
    def _heartbeat():
        while not stop_evt.wait(heartbeat_seconds):
            print(f'  ⏱  已執行 {_fmt_elapsed(time.monotonic() - start)}', flush=True)
    th = threading.Thread(target=_heartbeat, daemon=True)
    th.start()
    error = None
    try:
        yield
    except BaseException as e:
        error = e
        raise
    finally:
        stop_evt.set()
        th.join(timeout=1)
        total = _fmt_elapsed(time.monotonic() - start)
        if error is None:
            print(f'\n✓ {title} 完成，總耗時 {total}')
        else:
            print(f'\n✗ {title} 失敗（{type(error).__name__}），已執行 {total}')

print('✓ progress_stage 已就緒')


In [ ]:
# Cell 2: 安裝 LeRobot + SARM
import subprocess, sys

with progress_stage(
    '安裝 LeRobot + SARM（預計 2-5 分鐘）',
    steps=['pip install lerobot[sarm] from GitHub', '確認版本'],
    heartbeat_seconds=30,
):
    r = subprocess.run(
        'pip install -q "git+https://github.com/huggingface/lerobot.git#egg=lerobot[sarm]"',
        shell=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'pip install 失敗，返回碼 {r.returncode}')
    print('  ✓ LeRobot 安裝完成')

    check_cmd = 'import lerobot, transformers; print(lerobot.__version__); print(transformers.__version__)'
    r = subprocess.run([sys.executable, '-c', check_cmd], capture_output=True, text=True)
    if r.returncode == 0:
        v = r.stdout.strip().split()
        print(f'  LeRobot: {v[0]}, Transformers: {v[1] if len(v)>1 else "?"}')
    else:
        raise RuntimeError(f'版本確認失敗: {r.stderr}')


In [ ]:
# Cell 3: 修補 Transformers 5.x Bug（CRITICAL）
import os, glob, torch, subprocess
import lerobot

lerobot_root = os.path.dirname(lerobot.__file__)
hits = glob.glob(os.path.join(lerobot_root, '**', 'processor_sarm.py'), recursive=True)
if not hits:
    hits = glob.glob(os.path.join(os.path.dirname(lerobot_root), '**', 'processor_sarm.py'), recursive=True)
if not hits:
    raise FileNotFoundError('找不到 processor_sarm.py，請確認 lerobot[sarm] 安裝成功。')

processor_path = hits[0]
print(f'修補目標: {processor_path}')

with open(processor_path) as f:
    content = f.read()
patched = False
indent = '\n        '

for kind in ('image', 'text'):
    old = f'embeddings = self.clip_model.get_{kind}_features(**inputs).detach().cpu()'
    new = indent.join([
        f'output = self.clip_model.get_{kind}_features(**inputs)',
        'if not isinstance(output, torch.Tensor):',
        '    output = output.pooler_output',
        'embeddings = output.detach().cpu()',
    ])
    if old in content:
        content = content.replace(old, new)
        print(f'✓ 已修補 {kind} features')
        patched = True
    else:
        print(f'ℹ {kind} features 不需修補')

with open(processor_path, 'w') as f:
    f.write(content)

if patched:
    r = subprocess.run(['grep', '-n', 'pooler_output', processor_path],
                       capture_output=True, text=True)
    print('\n驗證修補（應看到 pooler_output）:')
    print(r.stdout)


In [ ]:
# Cell 4: HuggingFace 登入
from huggingface_hub import login
assert HF_TOKEN, '請先在 Cell 1 填入 HF_TOKEN'
login(token=HF_TOKEN)
print('✓ HuggingFace 登入成功')


In [ ]:
# Cell 5: dataset 修補三件套
# (A) 若沒有 v3.0 tag → 補 tag 或 monkey-patch get_safe_version
# (B) 修剪 info.json，只保留模型實際用到且 Hub 上有實檔的相機
# (C) 放寬 LeRobotDataset 預設 tolerance_s（避開 torchcodec FP 精度誤差）
import json, inspect
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
from lerobot.utils.constants import HF_LEROBOT_HUB_CACHE
import lerobot.datasets.lerobot_dataset as _lr_ds
import lerobot.datasets.utils as _ds_utils
import lerobot.datasets.dataset_metadata as _dsmeta

api = HfApi()

# --- (A) v3.0 tag ---
tag_ok = False
try:
    api.create_tag(ANNOTATED_DATASET, tag='v3.0', repo_type='dataset', exist_ok=True)
    print(f'✓ 已在 {ANNOTATED_DATASET} 上補打 v3.0 tag')
    tag_ok = True
except Exception as e:
    print(f'ℹ 無法補 tag（{type(e).__name__}），改用 monkey-patch fallback → "main"')

if not tag_ok:
    _orig_gsv = _ds_utils.get_safe_version
    def _patched_gsv(repo_id, version):
        try:
            return _orig_gsv(repo_id, version)
        except Exception as e:
            print(f'  ⚠ get_safe_version({repo_id}) 失敗（{type(e).__name__}），fallback → "main"')
            return 'main'
    _ds_utils.get_safe_version = _patched_gsv
    if hasattr(_dsmeta, 'get_safe_version'):
        _dsmeta.get_safe_version = _patched_gsv
    print('✓ get_safe_version monkey-patch 已就緒')

# --- (B) 修剪 info.json ---
MODEL_IMAGE_KEY = VIDEO_KEY
NEEDED_VIDEO_FEATURES = {MODEL_IMAGE_KEY}

meta_dir = Path(snapshot_download(
    repo_id=ANNOTATED_DATASET, repo_type='dataset', revision='main',
    allow_patterns=['meta/*'], cache_dir=HF_LEROBOT_HUB_CACHE,
))
info_path = meta_dir / 'meta' / 'info.json'

tree = api.list_repo_tree(
    repo_id=ANNOTATED_DATASET, repo_type='dataset',
    path_in_repo='videos', revision='main', recursive=False,
)
present_cams = {Path(e.path).name for e in tree if Path(e.path).name.startswith('observation.images')}

info = json.loads(info_path.read_text())
declared_cams = {k for k, v in info['features'].items()
                 if k.startswith('observation.images') and v.get('dtype') == 'video'}
keep_cams = declared_cams & present_cams & NEEDED_VIDEO_FEATURES
drop_cams = declared_cams - keep_cams
print(f'宣告: {sorted(declared_cams)}')
print(f'保留: {sorted(keep_cams)}')
print(f'移除: {sorted(drop_cams)}')
assert MODEL_IMAGE_KEY in keep_cams, f'⚠ {MODEL_IMAGE_KEY} 沒被保留'
if drop_cams:
    for cam in drop_cams:
        info['features'].pop(cam, None)
    info_path.write_text(json.dumps(info, indent=4))
    print(f'✓ 已從 info.json 移除 {len(drop_cams)} 個相機 feature')

# --- (C) tolerance_s ---
_OrigInit = _lr_ds.LeRobotDataset.__init__
_orig_sig = inspect.signature(_OrigInit)
def _patched_init(self, *args, **kwargs):
    kwargs.setdefault('tolerance_s', 1e-2)
    return _OrigInit(self, *args, **kwargs)
_patched_init.__signature__ = _orig_sig
_lr_ds.LeRobotDataset.__init__ = _patched_init
print('✓ LeRobotDataset 預設 tolerance_s 改為 1e-2')


In [ ]:
# Cell 6: 載入 SARM 資源（模型 + dataset + preprocess）— 只載一次，後面共用
from lerobot.rewards.sarm.compute_rabc_weights import load_sarm_resources

with progress_stage(
    f'載入 SARM 資源（model={MODEL_REPO_ID}）',
    steps=['SARMRewardModel.from_pretrained',
           'LeRobotDataset（帶 delta_timestamps）',
           'make_sarm_pre_post_processors'],
    heartbeat_seconds=30,
):
    dataset, reward_model, preprocess = load_sarm_resources(
        ANNOTATED_DATASET, MODEL_REPO_ID, device='cuda',
    )

# preprocess 進 eval 模式（關掉 augmentation）
if hasattr(preprocess, 'eval'):
    preprocess.eval()
for step in getattr(preprocess, 'steps', []):
    if hasattr(step, 'eval'):
        step.eval()

cfg = reward_model.config
image_key  = cfg.image_key
state_key  = cfg.state_key
target_idx = cfg.n_obs_steps // 2
dual_mode  = cfg.uses_dual_heads
head_mode  = HEAD_MODE if (HEAD_MODE in ('sparse', 'dense') and (HEAD_MODE == 'sparse' or dual_mode)) else 'sparse'
num_stages = getattr(cfg, f'num_{head_mode}_stages')
stage_labels_cfg = getattr(cfg, f'{head_mode}_subtask_names') or [f'Stage {i+1}' for i in range(num_stages)]

print(f'\nimage_key={image_key}, state_key={state_key}, target_idx={target_idx}')
print(f'head_mode={head_mode}, num_stages={num_stages}')
print(f'dataset.num_episodes = {dataset.num_episodes}')
print(f'subtask_names: {stage_labels_cfg}')


In [ ]:
# Cell 7: 生成「影片 + 同步進度條」MP4（in-process 推論）
import os
import numpy as np
import torch
from tqdm.auto import tqdm

from lerobot.rewards.sarm.compute_rabc_weights import (
    to_numpy_image, interpolate_progress,
)

os.makedirs(VIZ_VIDEO_DIR, exist_ok=True)


def render_episode_mp4(frames, progress_preds, stage_preds, title, mp4_path,
                       stage_labels, fps=30):
    """渲染左影片右進度條的 MP4。frames 為 ndarray (N, H, W, 3) uint8。"""
    import imageio
    from PIL import Image, ImageDraw, ImageFont
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    n = len(frames)
    if n == 0:
        return
    fh, fw = frames[0].shape[:2]
    panel_w = max(fw, 480)
    canvas_h = fh
    canvas_w = fw + panel_w

    dpi = 100
    fig, ax_p = plt.subplots(figsize=(panel_w / dpi, (canvas_h / dpi) * 0.55), dpi=dpi)
    fig.patch.set_facecolor('white')
    x = np.arange(n)
    ax_p.plot(x, progress_preds, color='#2E86AB', linewidth=2)
    ax_p.fill_between(x, 0, progress_preds, alpha=0.3, color='#2E86AB')
    ax_p.set_ylim(-0.05, 1.1)
    ax_p.set_xlim(0, max(n - 1, 1))
    ax_p.set_ylabel('Progress')
    ax_p.set_xlabel('Frame')
    ax_p.grid(True, alpha=0.3)
    ax_p.set_title('Predicted progress', fontsize=10)
    fig.tight_layout()
    fig.canvas.draw()
    if hasattr(fig.canvas, 'buffer_rgba'):
        curve_rgb = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    else:
        buf = fig.canvas.tostring_rgb()
        w_px, h_px = fig.canvas.get_width_height()
        curve_rgb = np.frombuffer(buf, dtype=np.uint8).reshape(h_px, w_px, 3)
    bbox = ax_p.get_window_extent()
    ax_x0_in = int(bbox.x0); ax_x1_in = int(bbox.x1)
    ax_y0_in = int(curve_rgb.shape[0] - bbox.y1)
    ax_y1_in = int(curve_rgb.shape[0] - bbox.y0)
    plt.close(fig)

    target_curve_h = int(canvas_h * 0.55)
    curve_img = Image.fromarray(curve_rgb).resize((panel_w, target_curve_h))
    sx = panel_w / curve_rgb.shape[1]
    sy = target_curve_h / curve_rgb.shape[0]
    ax_x0 = int(ax_x0_in * sx); ax_x1 = int(ax_x1_in * sx)
    ax_y0 = int(ax_y0_in * sy); ax_y1 = int(ax_y1_in * sy)

    try:
        font_tt = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 13)
        font_sm = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 12)
    except Exception:
        font_tt = font_sm = ImageFont.load_default()

    bar_top, bar_h = 12, 22
    bar_left, bar_right = 14, panel_w - 14
    title_y = bar_top + bar_h + 6
    curve_y0 = title_y + 30
    subtask_y = canvas_h - 40

    writer = imageio.get_writer(mp4_path, fps=fps, codec='libx264', quality=8,
                                macro_block_size=1)
    try:
        for i in range(n):
            canvas = Image.new('RGB', (canvas_w, canvas_h), 'white')
            vf = frames[i]
            if vf.ndim == 2:
                vf = np.stack([vf]*3, axis=-1)
            elif vf.shape[-1] == 1:
                vf = np.repeat(vf, 3, axis=-1)
            canvas.paste(Image.fromarray(vf.astype(np.uint8)), (0, 0))
            canvas.paste(curve_img, (fw, curve_y0))

            draw = ImageDraw.Draw(canvas)
            p = float(np.clip(progress_preds[i], 0, 1))

            draw.rectangle([(fw + bar_left, bar_top), (fw + bar_right, bar_top + bar_h)],
                           outline=(80, 80, 80), fill=(235, 235, 235), width=1)
            fill_x = fw + bar_left + int((bar_right - bar_left) * p)
            draw.rectangle([(fw + bar_left, bar_top), (fill_x, bar_top + bar_h)],
                           outline=None, fill=(46, 134, 171))
            draw.text((fw + bar_left, title_y), f'Progress: {p:.3f}',
                      fill=(0, 0, 0), font=font_tt)

            marker_x = fw + ax_x0 + int(i / max(n - 1, 1) * (ax_x1 - ax_x0))
            draw.line([(marker_x, curve_y0 + ax_y0), (marker_x, curve_y0 + ax_y1)],
                      fill=(0, 0, 0), width=2)
            dot_y = curve_y0 + ax_y1 - int((ax_y1 - ax_y0) * (p / 1.1 + 0.05/1.1))
            r = 4
            draw.ellipse([(marker_x - r, dot_y - r), (marker_x + r, dot_y + r)],
                         fill=(220, 40, 40), outline=(0, 0, 0))

            stage_idx = int(np.argmax(stage_preds[i]))
            label = stage_labels[stage_idx]
            if len(label) > 50:
                label = label[:50] + '...'
            draw.text((fw + bar_left, subtask_y), f'Subtask [{stage_idx}]: {label}',
                      fill=(0, 0, 0), font=font_sm)

            counter = f'Frame {i+1}/{n}'
            draw.rectangle((10, canvas_h - 28, 140, canvas_h - 6), fill=(0, 0, 0))
            draw.text((16, canvas_h - 24), counter, fill=(255, 255, 255), font=font_sm)

            title_short = title[:48] + ('...' if len(title) > 48 else '')
            draw.rectangle((6, 6, 6 + min(360, fw - 12), 28), fill=(0, 0, 0))
            draw.text((10, 8), title_short, fill=(255, 255, 255), font=font_sm)

            writer.append_data(np.array(canvas))
    finally:
        writer.close()


def run_inference_for_episode(episode_idx, stride):
    ep = dataset.meta.episodes[episode_idx]
    ep_start = int(ep['dataset_from_index'])
    ep_end   = int(ep['dataset_to_index'])
    task     = dataset[ep_start].get('task', 'perform the task')
    n_frames = ep_end - ep_start

    infer_local = list(range(0, n_frames, stride))
    if (n_frames - 1) not in infer_local:
        infer_local.append(n_frames - 1)
    infer_local = sorted(set(infer_local))

    prog = np.full(n_frames, np.nan, dtype=np.float32)
    stages = np.full((n_frames, num_stages), np.nan, dtype=np.float32)
    full_frames = [None] * n_frames

    for local_i in tqdm(range(n_frames), desc=f'Episode {episode_idx}', leave=False):
        f_idx = ep_start + local_i
        sample = dataset[f_idx]
        full_frames[local_i] = to_numpy_image(sample[image_key])
        if local_i not in infer_local:
            continue

        batch = {image_key: sample[image_key], 'task': task,
                 'index': f_idx, 'episode_index': episode_idx}
        if state_key in sample:
            batch[state_key] = sample[state_key]

        with torch.no_grad():
            processed = preprocess(batch)
            vf = processed['video_features'].to(reward_model.device)
            tf = processed['text_features'].to(reward_model.device)
            sf = processed.get('state_features')
            if sf is not None:
                sf = sf.to(reward_model.device)
            lengths = processed.get('lengths')
            reward, stage_probs = reward_model.calculate_rewards(
                text_embeddings=tf, video_embeddings=vf,
                state_features=sf, lengths=lengths,
                return_all_frames=True, return_stages=True,
                head_mode=head_mode,
            )
            if isinstance(reward, torch.Tensor):
                reward = reward.cpu().numpy()
                stage_probs = stage_probs.cpu().numpy()
            if reward.ndim == 2:
                prog[local_i] = reward[0, target_idx]
                stages[local_i] = stage_probs[0, target_idx, :]
            else:
                prog[local_i] = reward[target_idx]
                stages[local_i] = stage_probs[target_idx, :]
        torch.cuda.empty_cache()

    valid_idx = np.where(np.isfinite(prog))[0]
    all_idx = np.arange(n_frames)
    if valid_idx.size >= 1:
        prog_interp = interpolate_progress(valid_idx, prog[valid_idx], all_idx)
        stage_interp = np.zeros_like(stages)
        for s in range(num_stages):
            stage_interp[:, s] = interpolate_progress(valid_idx, stages[valid_idx, s], all_idx)
        row_sums = stage_interp.sum(axis=1, keepdims=True)
        nz = row_sums.squeeze(-1) > 0
        stage_interp[nz] = stage_interp[nz] / row_sums[nz]
    else:
        prog_interp = np.nan_to_num(prog, nan=0.0)
        stage_interp = np.nan_to_num(stages, nan=0.0)

    return np.stack(full_frames, axis=0), prog_interp, stage_interp, task


# === 主流程 ===
viz_episodes = [e for e in EVAL_EPISODES if 0 <= e < dataset.num_episodes]
skipped = [e for e in EVAL_EPISODES if e not in viz_episodes]
if skipped:
    print(f'⚠ 跳過超出範圍的 episode（dataset 共 {dataset.num_episodes} 個）: {skipped}')
if not viz_episodes:
    raise RuntimeError(f'EVAL_EPISODES 中沒有任何有效 episode（dataset 只有 {dataset.num_episodes} 個）')

with progress_stage(
    f'渲染 {len(viz_episodes)} 個 episode 的 MP4（stride={VIDEO_STRIDE}）',
    steps=[f'episodes={viz_episodes}',
           f'每個 episode 推論 + 抽幀 + 渲染',
           f'輸出到 {VIZ_VIDEO_DIR}'],
    heartbeat_seconds=30,
):
    for ep_idx in viz_episodes:
        full_frames, prog, stages_arr, task = run_inference_for_episode(ep_idx, VIDEO_STRIDE)
        mp4_path = os.path.join(VIZ_VIDEO_DIR, f'sarm_video_ep{ep_idx}_{head_mode}.mp4')
        render_episode_mp4(
            frames=full_frames,
            progress_preds=prog,
            stage_preds=stages_arr,
            title=f'{task} (ep {ep_idx})',
            mp4_path=mp4_path,
            stage_labels=stage_labels_cfg,
            fps=VIDEO_FPS,
        )
        print(f'  ✓ ep {ep_idx}: {mp4_path}')

print(f'\n所有 MP4 存放於：{VIZ_VIDEO_DIR}')


In [ ]:
# Cell 8: 在 notebook 內嵌入剛剛產生的 MP4
import glob, os
from IPython.display import Video, display

video_files = sorted(glob.glob(f'{VIZ_VIDEO_DIR}/*.mp4'))
if not video_files:
    print('找不到 MP4，請確認 Cell 7 成功執行')
else:
    print(f'找到 {len(video_files)} 支 MP4：')
    for p in video_files:
        print(f'\n{os.path.basename(p)} ({os.path.getsize(p)/1e6:.1f} MB):')
        display(Video(p, embed=True, width=900))


In [ ]:
# Cell 9: 打包下載
import os, shutil, glob

zip_base = '/content/sarm_predictions_video'
mp4_files = sorted(glob.glob(f'{VIZ_VIDEO_DIR}/*.mp4'))
if not mp4_files:
    raise RuntimeError(f'{VIZ_VIDEO_DIR} 內沒有 MP4，請先跑 Cell 7')

total_mb = 0
print(f'準備打包 {len(mp4_files)} 支影片：')
for p in mp4_files:
    size_mb = os.path.getsize(p) / 1e6
    total_mb += size_mb
    print(f'  - {os.path.basename(p)}  ({size_mb:.1f} MB)')
print(f'總大小: {total_mb:.1f} MB')

zip_path = shutil.make_archive(zip_base, 'zip', root_dir=VIZ_VIDEO_DIR)
print(f'\n✓ 已打包: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')

try:
    from google.colab import files
    files.download(zip_path)
    print('✓ 已觸發瀏覽器下載')
except ImportError:
    print('（不是 Colab 環境，跳過 files.download）')

# 備案：存到 Google Drive（大檔建議）
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(zip_path, '/content/drive/MyDrive/sarm_predictions_video.zip')
